# Lab 8 — 1000-State Random Walk: Algorithms + Final Plots

This notebook reorganizes the Lab 8 material so that:

- environment and value-function representations are defined first;
- each learning algorithm is written as its own function/cell;
- **all experiments, plot commands, and plotting code are kept together in the final cell**;
- the algorithms themselves contain no plotting code.

Algorithms covered:
1. Exact policy evaluation / dynamic programming reference value
2. Gradient Monte Carlo
3. Semi-gradient n-step TD

Value-function representations covered:
- State aggregation
- Tile coding
- Polynomial basis
- Fourier basis


## 1. Setup

The environment has 1000 non-terminal states, numbered 1–1000, with terminal states 0 and 1001. The agent starts at state 500, chooses left/right with probability 0.5, and moves 1–100 states in the selected direction.

The terminal rewards are −1 on the left and +1 on the right, with γ = 1.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

SEED = 2026
np.random.seed(SEED)

FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

N_STATES = 1000
STATES = np.arange(1, N_STATES + 1)
START_STATE = 500
END_STATES = [0, N_STATES + 1]
ACTION_LEFT, ACTION_RIGHT = -1, 1
ACTIONS = [ACTION_LEFT, ACTION_RIGHT]
STEP_RANGE = 100
GAMMA = 1.0

print('setup complete')

## 2. Environment functions

The transition is stochastic because the stride is uniformly sampled from 1 to 100.

In [ ]:
def step(state, action):
    stride = np.random.randint(1, STEP_RANGE + 1)
    state += stride * action
    state = max(min(state, N_STATES + 1), 0)

    if state == 0:
        reward = -1
    elif state == N_STATES + 1:
        reward = 1
    else:
        reward = 0

    return state, reward


def get_action():
    return 1 if np.random.binomial(1, 0.5) == 1 else -1

## 3. Algorithm 1 — Exact policy evaluation / DP reference

For the random policy, the Bellman expectation equation is

$$V(s) = \frac{1}{2\cdot100}\sum_{a\in\{-1,+1\}}\sum_{u=1}^{100}V\left(\operatorname{clip}(s+au)\right).$$

This is used only as the reference value function for measuring RMS error.

In [ ]:
def compute_true_value():
    v = np.arange(-1001, 1003, 2) / 1001.0

    idx = np.empty((2 * STEP_RANGE, N_STATES + 2), dtype=int)
    k = 0

    for action in ACTIONS:
        for stride in range(1, STEP_RANGE + 1):
            idx[k] = np.clip(
                np.arange(N_STATES + 2) + stride * action,
                0,
                N_STATES + 1
            )
            k += 1

    idx = idx[:, STATES]
        
    sweeps = 0
    while True:
        old = v.copy()
        v[STATES] = v[idx].mean(axis=0)
        sweeps += 1

        if np.abs(old - v).sum() < 1e-2:
            break

    v[0] = v[-1] = 0.0
    return v, sweeps

## 4. Value-function representations

All learning algorithms use the same basic update structure

$$\mathbf{w}\leftarrow\mathbf{w}+\alpha\delta\nabla_{\mathbf w}\hat v(s,\mathbf w),$$

but the representation determines the gradient/features.

In [ ]:
class ValueFunction:
    def __init__(self, num_of_groups):
        self.num_of_groups = num_of_groups
        self.group_size = N_STATES // num_of_groups
        self.params = np.zeros(num_of_groups)

    def value(self, state):
        if state in END_STATES:
            return 0.0
        return self.params[(state - 1) // self.group_size]

    def update(self, delta, state):
        self.params[(state - 1) // self.group_size] += delta


class TilingsValueFunction:
    def __init__(self, numOfTilings, tileWidth, tilingOffset):
        self.numOfTilings = numOfTilings
        self.tileWidth = tileWidth
        self.tilingOffset = tilingOffset
        self.tilingSize = N_STATES // tileWidth + 1
        self.params = np.zeros((self.numOfTilings, self.tilingSize))
        self.tilings = np.arange(-tileWidth + 1, 0, tilingOffset)

    def value(self, state):
        state_value = 0.0
        for t in range(len(self.tilings)):
            tile_index = (state - self.tilings[t]) // self.tileWidth
            state_value += self.params[t, tile_index]
        return state_value

    def update(self, delta, state):
        delta /= self.numOfTilings
        for t in range(len(self.tilings)):
            tile_index = (state - self.tilings[t]) // self.tileWidth
            self.params[t, tile_index] += delta


POLYNOMIAL_BASES, FOURIER_BASES = 0, 1


class BasesValueFunction:
    def __init__(self, order, kind):
        self.order = order
        self.kind = kind
        self.weights = np.zeros(order + 1)
        self.bases = []

        if kind == POLYNOMIAL_BASES:
            for i in range(order + 1):
                self.bases.append(lambda s, i=i: s ** i)
        elif kind == FOURIER_BASES:
            for i in range(order + 1):
                self.bases.append(lambda s, i=i: np.cos(i * np.pi * s))

    def value(self, state):
        state = state / float(N_STATES)
        features = np.asarray([f(state) for f in self.bases])
        return np.dot(self.weights, features)

    def update(self, delta, state):
        state = state / float(N_STATES)
        features = np.asarray([f(state) for f in self.bases])
        self.weights += delta * features

In [ ]:
def values_all_states_agg(vf):
    return np.repeat(vf.params, vf.group_size)


def values_all_states_tiling(vf):
    s = np.arange(1, N_STATES + 1)[:, None]
    tile_idx = (s - vf.tilings[None, :]) // vf.tileWidth
    return vf.params[
        np.arange(vf.numOfTilings)[None, :],
        tile_idx
    ].sum(axis=1)


def basis_matrix(vf):
    s = np.arange(1, N_STATES + 1) / float(N_STATES)
    return np.asarray([[f(si) for f in vf.bases] for si in s])

## 5. Algorithm 2 — Gradient Monte Carlo

Monte Carlo waits until the episode terminates and uses the complete return $G_t$.

$$\delta = \alpha\left(G_t-\hat v(S_t,\mathbf w)\right)$$

and then

$$\mathbf w\leftarrow\mathbf w+\alpha\left(G_t-\hat v(S_t,\mathbf w)\right)\nabla_{\mathbf w}\hat v(S_t,\mathbf w).$$

For this random walk, γ = 1 and only the terminal reward is non-zero, so the final terminal reward is the return for every visited non-terminal state.

In [ ]:
def gradient_monte_carlo(value_function, alpha, distribution=None):
    state = START_STATE
    trajectory = [state]
    reward = 0.0

    while state not in END_STATES:
        next_state, reward = step(state, get_action())
        trajectory.append(next_state)
        state = next_state

    for state in trajectory[:-1]:
        delta = alpha * (reward - value_function.value(state))
        value_function.update(delta, state)

        if distribution is not None:
            distribution[state] += 1

## 6. Algorithm 3 — Semi-gradient n-step TD

The n-step target is

$$G_{t:t+n}=R_{t+1}+\gamma R_{t+2}+\cdots+\gamma^{n-1}R_{t+n}+\gamma^n\hat v(S_{t+n},\mathbf w).$$

If the episode terminates before $t+n$, the bootstrap term disappears and the target becomes the Monte Carlo return.

In [ ]:
def semi_gradient_temporal_difference(value_function, n, alpha):
    state = START_STATE
    states = [state]
    rewards = [0]
    T = float('inf')
    t_now = 0

    while True:
        t_now += 1

        if t_now < T:
            next_state, reward = step(state, get_action())
            states.append(next_state)
            rewards.append(reward)

            if next_state in END_STATES:
                T = t_now

        update_time = t_now - n

        if update_time >= 0:
            returns = 0.0

            for t in range(
                update_time + 1,
                min(T, update_time + n) + 1
            ):
                returns += rewards[t]

            if update_time + n <= T:
                returns += value_function.value(states[update_time + n])

            state_to_update = states[update_time]

            if state_to_update not in END_STATES:
                delta = alpha * (
                    returns - value_function.value(state_to_update)
                )
                value_function.update(delta, state_to_update)

        if update_time == T - 1:
            break

        state = next_state

## 7. Final cell — experiments + all plots

Everything below is intentionally kept in **one final cell**. Running this cell generates the reference-value convergence plot, n-step TD RMS-vs-α plot, gradient-MC state-aggregation plot, tile-coding comparison, and polynomial-vs-Fourier comparison.

In [ ]:
# ============================================================
# FINAL CELL: ALL EXPERIMENTS + ALL PLOT COMMANDS
# ============================================================

# ---------- reference value ----------
t0 = time.time()
true_value, n_sweeps = compute_true_value()
print(f"DP reference converged in {n_sweeps} sweeps ({time.time() - t0:.2f}s)")
print(
    "value at state 1 / 500 / 1000:",
    round(true_value[1], 3),
    round(true_value[500], 3),
    round(true_value[1000], 3)
)


# ---------- Plot 1: true-value convergence ----------
def compute_true_value_trace(snapshots=(1, 2, 5, 10, 50)):
    v = np.arange(-1001, 1003, 2) / 1001.0

    idx = np.empty((2 * STEP_RANGE, N_STATES + 2), dtype=int)
    k = 0

    for action in ACTIONS:
        for stride in range(1, STEP_RANGE + 1):
            idx[k] = np.clip(
                np.arange(N_STATES + 2) + stride * action,
                0,
                N_STATES + 1
            )
            k += 1

    idx = idx[:, STATES]

    trace = {}
    sweeps = 0

    while True:
        old = v.copy()
        v[STATES] = v[idx].mean(axis=0)
        sweeps += 1

        if sweeps in snapshots:
            trace[sweeps] = v.copy()

        if np.abs(old - v).sum() < 1e-2:
            break

    v[0] = v[-1] = 0.0
    return v, trace, sweeps


v_final, trace, sweeps = compute_true_value_trace()

fig, ax = plt.subplots(figsize=(8, 4.4), constrained_layout=True)

for n_sweep, v_snap in trace.items():
    ax.plot(
        STATES,
        v_snap[1:-1],
        lw=1.2,
        label=f"after {n_sweep} sweep(s)"
    )

ax.plot(
    STATES,
    v_final[1:-1],
    lw=2.2,
    label=f"converged ({sweeps} sweeps)"
)

ax.set_xlabel("State")
ax.set_ylabel("Value")
ax.set_title("Dynamic Programming Value Convergence")
ax.legend(loc="upper left")

fig.savefig(f"{FIG_DIR}/fig_true_value_convergence.png")
plt.show()


# ---------- Plot 2: n-step TD RMS error vs alpha ----------
steps = [1, 2, 4, 8, 16, 32]
alphas = np.arange(0, 1.1, 0.1)

episodes = 10
runs = 10

errors = np.zeros((len(steps), len(alphas)))

t0 = time.time()

for run in range(runs):
    for si, n in enumerate(steps):
        for ai, alpha in enumerate(alphas):
            vf = ValueFunction(20)

            for ep in range(episodes):
                semi_gradient_temporal_difference(vf, n, alpha)

                sv = values_all_states_agg(vf)

                errors[si, ai] += np.sqrt(
                    np.sum((sv - true_value[1:-1]) ** 2) / N_STATES
                )

    print(f"n-step run {run + 1}/{runs} finished ({time.time() - t0:.1f}s)")

errors /= episodes * runs

fig, ax = plt.subplots(figsize=(8, 4.6), constrained_layout=True)

for si, n in enumerate(steps):
    ax.plot(
        alphas,
        errors[si],
        label=f"n = {n}"
    )

ax.set_xlabel(r"learning rate $\alpha$")
ax.set_ylabel("RMS error (averaged over episodes)")
ax.set_ylim(0.25, 0.55)
ax.set_xticks(alphas)
ax.legend(ncol=3, loc="lower right")

fig.savefig(f"{FIG_DIR}/fig_nstep_rms_vs_alpha.png")
plt.show()

best = np.unravel_index(
    np.nanargmin(np.where(errors > 0, errors, np.inf)),
    errors.shape
)

print(
    f"best n-step setting: n = {steps[best[0]]}, "
    f"alpha = {alphas[best[1]]:.1f}, "
    f"RMS = {errors[best]:.3f}"
)


# ---------- Plot 3: Gradient MC + state aggregation ----------
episodes = 20000
alpha = 1e-4

vf = ValueFunction(10)
distribution = np.zeros(N_STATES + 2)

t0 = time.time()

for ep in range(episodes):
    gradient_monte_carlo(vf, alpha, distribution)

    if (ep + 1) % 5000 == 0:
        print(
            f"MC episode {ep + 1:>6}/{episodes} "
            f"({time.time() - t0:.1f}s)"
        )

distribution /= distribution.sum()

sv = values_all_states_agg(vf)

rms = np.sqrt(
    np.mean((sv - true_value[1:-1]) ** 2)
)

print(f"final MC RMS error: {rms:.4f}")

fig, axes = plt.subplots(
    2,
    1,
    figsize=(8, 7.2),
    constrained_layout=True,
    sharex=True
)

axes[0].plot(
    STATES,
    sv,
    lw=2.0,
    label="Approximate MC value (10 groups)"
)

axes[0].plot(
    STATES,
    true_value[1:-1],
    lw=1.6,
    label="True value (DP)"
)

axes[0].axvline(500, lw=0.8, ls=":")
axes[0].set_ylabel("Value")
axes[0].set_title(
    "Gradient Monte Carlo with State Aggregation"
)
axes[0].legend(loc="upper left")

axes[1].plot(
    STATES,
    distribution[1:-1],
    lw=1.6
)

axes[1].set_xlabel("State")
axes[1].set_ylabel("State distribution")

fig.savefig(f"{FIG_DIR}/fig_mc_state_aggregation.png")
plt.show()


# ---------- Plot 4: Tile coding vs state aggregation ----------
episodes = 5000

num_of_tilings = 50
tile_width = 200
tiling_offset = 4

labels = [
    "Tile coding (50 tilings)",
    "State aggregation (one tiling, 5 groups)"
]

vfs = [
    TilingsValueFunction(
        num_of_tilings,
        tile_width,
        tiling_offset
    ),
    ValueFunction(N_STATES // tile_width)
]

errors = np.zeros((2, episodes))

t0 = time.time()

for i, vf in enumerate(vfs):

    for ep in range(episodes):

        alpha = 1.0 / (ep + 1)

        gradient_monte_carlo(vf, alpha)

        if i == 0:
            sv = values_all_states_tiling(vf)
        else:
            sv = values_all_states_agg(vf)

        errors[i, ep] = np.sqrt(
            np.mean(
                (true_value[1:-1] - sv) ** 2
            )
        )

    print(
        f"{labels[i]}: final RMS = {errors[i, -1]:.4f} "
        f"({time.time() - t0:.1f}s)"
    )

fig, ax = plt.subplots(
    figsize=(8, 4.4),
    constrained_layout=True
)

ax.plot(
    errors[0],
    lw=2.0,
    label=labels[0]
)

ax.plot(
    errors[1],
    lw=2.0,
    label=labels[1]
)

ax.set_xlabel("Episodes")
ax.set_ylabel("RMS error")
ax.set_title("Tile Coding vs State Aggregation")
ax.legend(loc="upper right")

fig.savefig(f"{FIG_DIR}/fig_tile_coding_vs_aggregation.png")
plt.show()


# ---------- Plot 5: Polynomial vs Fourier bases ----------
runs = 1
episodes = 5000

orders = [5, 10, 20]

alphas = [
    1e-4,
    5e-5
]

kind_labels = [
    "polynomial",
    "fourier"
]

errors = np.zeros(
    (2, len(orders), episodes)
)

t0 = time.time()

for oi, order in enumerate(orders):

    vfs = [
        BasesValueFunction(
            order,
            POLYNOMIAL_BASES
        ),
        BasesValueFunction(
            order,
            FOURIER_BASES
        )
    ]

    mats = [
        basis_matrix(vfs[0]),
        basis_matrix(vfs[1])
    ]

    for j, vf in enumerate(vfs):

        F = mats[j]

        for ep in range(episodes):

            gradient_monte_carlo(
                vf,
                alphas[j]
            )

            sv = F @ vf.weights

            errors[j, oi, ep] = np.sqrt(
                np.mean(
                    (true_value[1:-1] - sv) ** 2
                )
            )

    print(
        f"order {order}: "
        f"poly RMS = {errors[0, oi, -1]:.3f}, "
        f"fourier RMS = {errors[1, oi, -1]:.3f} "
        f"({time.time() - t0:.1f}s)"
    )


fig, ax = plt.subplots(
    figsize=(8.4, 4.8),
    constrained_layout=True
)

for j, kind in enumerate(kind_labels):

    for oi, order in enumerate(orders):

        ax.plot(
            errors[j, oi],
            lw=2.0 if oi == 0 else 1.4 if oi == 1 else 1.0,
            label=f"{kind}, order = {order}"
        )

ax.set_xlabel("Episodes")
ax.set_ylabel("RMS error")
ax.set_title("Polynomial vs Fourier Basis Functions")
ax.legend(ncol=2, fontsize=9, loc="upper right")

fig.savefig(f"{FIG_DIR}/fig_polynomial_vs_fourier.png")
plt.show()


print("\nAll experiments and plots completed.")
print(f"Figures saved in: {FIG_DIR}/")
